In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import base64

load_dotenv()

True

In [2]:
with open("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

image_b64[:200]

'iVBORw0KGgoAAAANSUhEUgAAAmwAAAHgCAIAAACXbaZMAACxv0lEQVR4nOzdeVwT1944/iGBkASIIKuyBBAQIahYqiBuiCjWrVi8eK+KYlVEqdvFBQtq64obVvsgUhatD9XSihuLFatYZRNUtA0iYF0AZZEiEBICSeb3ejy/O9+52QiRzfp5/5WcOXPmzDnDfJiZkzka'

In [3]:
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
    {"type": "text", "text": "This is a blood work report. Extract all test results and flag any values outside the normal range"}
])

response = llm.invoke([message])
print(response.content)

**Extracted Test Results with Abnormal Values Flagged**

### COMPLETE BLOOD COUNT (CBC)

* Hemoglobin: **15.1 g/dL** (Normal: 13.5-17.5) **(Within Normal Range)**
* Hematocrit: **44%** (Normal: 41-53) **(Within Normal Range)**
* WBC: **6.8 x10^3/uL** (Normal: 4.5-11.0) **(Within Normal Range)**
* Platelets: **220 x10^3/uL** (Normal: 150-400) **(Within Normal Range)**

### LIPID PANEL

* Total Cholesterol: **238 mg/dL** (Normal: <200) **(Elevated)**
* LDL Cholesterol: **162 mg/dL** (Normal: <100) **(Elevated)**
* HDL Cholesterol: **36 mg/dL** (Normal: >40) **(Low)**
* Triglycerides: **188 mg/dL** (Normal: <150) **(Elevated)**

### METABOLIC PANEL

* Glucose (Fasting): **92 mg/dL** (Normal: 70-99) **(Within Normal Range)**
* HbA1c: **5.3%** (Normal: <5.7) **(Within Normal Range)**
* Creatinine: **1.0 mg/dL** (Normal: 0.7-1.3) **(Within Normal Range)**
* eGFR: **82 mL/min** (Normal: >60) **(Within Normal Range)**

### LIVER FUNCTION

* No results provided.

**Summary of Abnormal Values:**

In [6]:
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

@tool
def get_diet_recommendation(image_path: str) -> str:
    """Analyze a blood work image and return a diet plan."""
    with open(image_path, "rb") as f:
        image_b64 = base64.b64encode(f.read()).decode()

    prompt = (
        "This is a blood work report. Extract all test results, flag any values "
        "outside the normal range, and provide a detailed diet plan that addresses "
        "the abnormalities. Include foods to eat, foods to avoid, and general advice."
    )
    message = HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text", "text": prompt}
    ])
    return llm.invoke([message]).content

agent = create_agent(
    llm,
    tools=[get_diet_recommendation],
    system_prompt=(
        "You are a helpful medical and nutrition assistant. "
        "When a user asks for a diet plan and mentions a blood report image, "
        "you MUST call the 'get_diet_recommendation' tool with the file path. "
        "Do not make up information."
    ),
    checkpointer=InMemorySaver()
)

user_msg = "Here is my blood work: blood_work.png. What diet should I follow?"
config = {"configurable": {"thread_id": "user1"}}
response = agent.invoke({"messages": [HumanMessage(content=user_msg)]}, config)
print(response["messages"][-1].content)

Based on the blood work provided, here's a concise diet plan:

**Focus on:**
- Soluble fiber (oatmeal, fruits, vegetables)
- Fatty fish (salmon, mackerel)
- Healthy fats (avocado, olive oil)
- Lean protein (chicken, legumes)

**Limit:**
- Saturated and trans fats (red meat, processed foods)
- High cholesterol foods (egg yolks, organ meats)
- Added sugars (sugary drinks, refined grains)

**Example Meal Plan:**
- Breakfast: Oatmeal with fruits and nuts
- Lunch: Grilled chicken salad with avocado
- Dinner: Baked salmon with quinoa and steamed vegetables

Consult a healthcare provider or registered dietitian for personalized guidance.
